# Standalone SEXTANTS hyperspectral physical-projection imaging

Run this notebook directly from raw NeXus files and beamstop PNGs. No FTH calibration,
series result, support HDF5, or execution of another notebook is required. All acquisition,
centering, dark, focus, support and thickness choices are defined here. The physical engine
is included under `library/hyperspectral_engine`, with no parent-repository dependency.

The joint model is `log(object) = common_illumination + thickness * (charge(E) + helicity * magnetic(E) * magnetization)`.
By default the algorithm fits nonnegative relative thickness throughout the whole support,
including reference apertures; their zero-thickness classification is not prescribed.
Charge/magnetic spectra and magnetization remain relative without external scale anchors.


In [ ]:
from pathlib import Path
import sys
from importlib import reload
import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = Path.cwd()
sys.path.insert(0, str(BASEFOLDER / "library"))
import fth_phase_workflow as wf
import sextants_hyperspectral as hs
hs = reload(hs)
print("Local engine:", hs.engine.__file__)


## All settings and recipe parameters

Edit this opening cell before loading data. It contains the acquisition/preprocessing
settings and **all 160 engine recipe controls**, grouped by purpose. `None` and `AUTO`
are deliberate visible choices, not hidden default values. The four AUTO bindings need
the raw dataset and are resolved immediately after loading. Inactive controls for other
engine model families are retained and labeled; listing them does not enable those models.


In [ ]:
USER = "rb"
# RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20261110_Battistelli_laser_RF/"
RAW_FOLDER = "/media/riccardo/EXT_20TB/DATA/2026/COMET_20261110_Battistelli_laser_RF"
MASK_FOLDER = BASEFOLDER / "processed" / "mask_pixels"
# Standalone starting values: verify against the raw preview below.
DETECTOR_CENTER = (962.0, 894.0)  # (row, column); per-pair "center" overrides are allowed.
PIXEL_SIZE_M = 11.0e-6
REFERENCE_ENERGY_EV = None  # None reads the first positive raw file's photon energy.
REFERENCE_DISTANCE_M = None  # None reads its NeXus CCD translation/distance metadata.
FOCUS = {"prop_dist": 28.55, "phase": -0.56, "dx": -0.24, "dy": 0.0}  # um, rad, detector pixels.
DARK_FIT_RECIPE = {
    "rows": [0, 600],  # Start and exclusive stop for the detector rows used in the dark fit.
    "columns": [0, 200],  # Start and exclusive stop for the detector columns used in the dark fit.
    "percentile": 100,  # Upper percentile cutoff for pixels included in the dark fit.
    "stride": 1,  # Pixel sampling step within the fit region.
}

ENERGIES_EV = [775, 777, 779, 780, 781, 782, 783, 783.5, 783.75, 784,
               784.25, 784.5, 785, 786, 787, 788, 790, 792, 794, 796]
PLUS_IMAGE_IDS = list(range(1797, 1837, 2))
MINUS_IMAGE_IDS = [image_id + 1 for image_id in PLUS_IMAGE_IDS]
assert len(PLUS_IMAGE_IDS) == len(MINUS_IMAGE_IDS) == len(ENERGIES_EV)
PAIRS = [dict(plus=plus, minus=minus, energy_eV=energy, state="sample", illumination="beam0")
         for plus, minus, energy in zip(PLUS_IMAGE_IDS, MINUS_IMAGE_IDS, ENERGIES_EV)]
# Example average: PAIRS[0]["plus"] = [1797, another_plus_scan_at_775eV]
DARK_IDS = {"+": 1468, "-": 1468}  # Raw dark scans: verify for this camera/beamstop session.
BEAMSTOP_MASK_IDS = {"+": 1795, "-": 1795}  # Explicit raw-coordinate beamstop PNG identifiers.
# Per-pair overrides: PAIRS[i]["dark_ids"] / ["beamstop_mask_ids"] / ["center"].
# Optional incoming-flux monitor: PAIRS[i]["incident_intensity"] = positive_monitor_value

SUPPORT_PNG = None  # Optional manually prepared support PNG: red pixels are INCLUDED.
SUPPORT_COORDINATES = [(1024, 1024, 1.0), (789, 1248, 76.0)]  # (row, column, radius), centered object grid.
FIT_MATERIAL_THICKNESS = True
MATERIAL_REGION = "support"  # "support" lets thickness fit all apertures; "largest_aperture" supplies a stronger prior.
INITIAL_MATERIAL_THICKNESS = None  # None starts positive throughout the allowed region.
ENERGY_TOLERANCE_EV = 0.35
EXPOSURE_NORMALIZE = True
INTENSITY_OFFSET = 0.0  # Explicit detector-intensity offset, applied before amplitudes.
RAW_DTYPE = "float32"  # Prepared intensity cube storage; the solver uses complex fields.
COMMON_Q_GRID = True
BINNING = 1  # Start with 1. Binning/crop must retain both sample and reference apertures.
CROP = 0
MODES = [1, 2]
PARTIAL_COHERENCE = False  # Enable only after checking the coherent fit.
OVERWRITE = False
SAVE_PREPARED = True

IM_ID = hs.first_id(PAIRS[0]["plus"])
PREPARED_H5 = BASEFOLDER / "processed" / "Logs" / f"hyperspectral_prepared_ImId_{IM_ID:04d}_{USER}.hdf5"
RESULT_H5 = BASEFOLDER / "processed" / "Logs" / f"hyperspectral_physical_ImId_{IM_ID:04d}_{USER}.hdf5"
FIGURE_PNG = BASEFOLDER / "processed" / f"hyperspectral_physical_ImId_{IM_ID:04d}_{USER}.png"
for row in PAIRS:
    print(row["energy_eV"], "eV: +", row["plus"], " / -", row["minus"])

# Every engine recipe parameter is explicit here. No defaults are filled in later.
# AUTO is permitted only for projection_every, energy_values, material_mask and projection_focus_setup.
AUTO = hs.AUTO_FROM_DATASET
RECIPE_PARAMETERS = {

    # Iteration schedules and initialization
    'inner_mode': ["ER"],  # Ordered phase-retrieval algorithms run per observation update.
    'inner_Nit': [50 if PARTIAL_COHERENCE else 5],  # Iteration count for each inner algorithm, in the same order as inner_mode.
    'outer_iterations': 50,  # Number of joint observation sweeps. Zero runs warmup and configured final constraints.
    'warmup_mode': ["HAPRE", "ER"],  # Ordered algorithms for independent initialization of each observation.
    'warmup_Nit': [700, 90],  # Iteration count for each warmup algorithm, in the same order as warmup_mode.
    'warmup_reference_mode': None,  # Algorithm override for reference warmup. None uses warmup_mode.
    'warmup_reference_Nit': None,  # Iteration counts for the reference warmup override. Set together with its modes.
    'warmup_other_mode': None,  # Algorithm override for non-reference warmups. None uses warmup_mode.
    'warmup_other_Nit': None,  # Iteration counts for non-reference warmup overrides. Set together with their modes.
    'warmup_start_from_first': False,  # Seed the other observations from the warmed-up reference field.
    'warmup_reference_observation': 0,  # Zero-based index of the observation used as the warmup reference.
    'startimage_scale_fit': 'linear',  # Method used to match an initial field's predicted intensity to measured intensity.
    'startimage_radial_normalization': False,  # Radially normalize the starting-image intensity for the scale fit.
    'warmup_seed_scale_fit': 'sum',  # Intensity scaling method used when transferring a warmup seed between observations.
    'warmup_seeded_stage_indices': None,  # Zero-based warmup stages to run on seeded observations. None runs the full schedule.
    'freeze_saturated_fields': False,  # Hold fields of explicitly saturated states fixed after warmup.

    # Partial coherence and shared-kernel behavior
    'partial_coherence': PARTIAL_COHERENCE,  # Enable partial-coherence modeling with fitted diffraction blur kernels.
    'coherence_kernel_scope': 'per_observation',  # Use one kernel per observation or one shared kernel across observations.
    'shared_coherence_update': 'sequential',  # sequential, average, or pooled; average/pooled require preserved warmup fills in this engine.
    'warmup_calibrate_shared_gamma': False,  # Calibrate the shared coherence kernel during warmup.
    'coherence_round_iterations': 50,  # Number of Richardson-Lucy iterations in a shared-kernel round update.
    'coherent_refresh_rounds': [],  # One-based outer rounds that run a coherent refresh before partial-coherence updates.
    'coherent_refresh_iterations': 50,  # Number of phase-retrieval iterations per coherent refresh.
    'coherent_refresh_mode': 'ER',  # Phase-retrieval algorithm used during a coherent refresh.
    'preserve_warmup_masked_intensity': False,  # False keeps missing pixels free; True intentionally fixes synthetic coherent fills.
    'freeze_coherence_after_reference': False,  # Keep the shared kernel fixed after calibration on the reference observation.

    # Execution, ordering and reproducibility
    'observation_workers': 1,  # Number of workers for eligible parallel observation updates.
    'fft_workers': 1,  # Number of workers used by FFT operations.
    'shuffle_observations': True,  # Randomize observation update order within joint sweeps.
    'random_seed': 0,  # Seed for randomized observation ordering. None leaves the seed unspecified.

    # Joint-stage numerical controls
    'beta_zero': 0.5,  # Base feedback coefficient for the beta schedule.
    'beta_mode': 'arctan',  # Schedule that varies the phase-retrieval feedback coefficient over a stage.
    'alpha_zero': 0.0,  # Base total-variation regularization strength. Zero disables its effect.
    'alpha_mode': 'const',  # Schedule for the total-variation regularization strength over a stage.
    'TV_freq': 1000000000.0,  # Interval in phase-retrieval iterations between total-variation updates.
    'RL_it': 20 if PARTIAL_COHERENCE else 0,  # Number of Richardson-Lucy iterations per coherence-kernel update. Zero disables these updates.
    'RL_freq': 20,  # Interval in phase-retrieval iterations between Richardson-Lucy updates.

    # Warmup controls (None inherits corresponding shared controls)
    'warmup_beta_zero': None,  # Shared warmup base feedback coefficient. None inherits beta_zero.
    'warmup_beta_mode': None,  # Shared warmup feedback coefficient schedule. None inherits beta_mode.
    'warmup_alpha_zero': None,  # Shared warmup base total-variation strength. None inherits alpha_zero.
    'warmup_alpha_mode': None,  # Shared warmup total-variation strength schedule. None inherits alpha_mode.
    'warmup_TV_freq': None,  # Shared warmup total-variation update interval in solver iterations. None inherits TV_freq.
    'warmup_RL_it': [0, 0],  # Shared warmup Richardson-Lucy iteration count per kernel update. None inherits RL_it.
    'warmup_RL_freq': [1e9, 1e9],  # Shared warmup Richardson-Lucy update interval in solver iterations. None inherits RL_freq.
    'warmup_reference_beta_zero': None,  # Reference warmup base feedback coefficient. None inherits warmup_beta_zero.
    'warmup_reference_beta_mode': None,  # Reference warmup feedback coefficient schedule. None inherits warmup_beta_mode.
    'warmup_reference_alpha_zero': None,  # Reference warmup base total-variation strength. None inherits warmup_alpha_zero.
    'warmup_reference_alpha_mode': None,  # Reference warmup total-variation strength schedule. None inherits warmup_alpha_mode.
    'warmup_reference_TV_freq': None,  # Reference warmup total-variation update interval in solver iterations. None inherits warmup_TV_freq.
    'warmup_reference_RL_it': None,  # Reference warmup Richardson-Lucy iteration count per kernel update. None inherits warmup_RL_it.
    'warmup_reference_RL_freq': None,  # Reference warmup Richardson-Lucy update interval in solver iterations. None inherits warmup_RL_freq.
    'warmup_other_beta_zero': None,  # Non-reference warmup base feedback coefficient. None inherits warmup_beta_zero.
    'warmup_other_beta_mode': None,  # Non-reference warmup feedback coefficient schedule. None inherits warmup_beta_mode.
    'warmup_other_alpha_zero': None,  # Non-reference warmup base total-variation strength. None inherits warmup_alpha_zero.
    'warmup_other_alpha_mode': None,  # Non-reference warmup total-variation strength schedule. None inherits warmup_alpha_mode.
    'warmup_other_TV_freq': None,  # Non-reference warmup total-variation update interval in solver iterations. None inherits warmup_TV_freq.
    'warmup_other_RL_it': None,  # Non-reference warmup Richardson-Lucy iteration count per kernel update. None inherits warmup_RL_it.
    'warmup_other_RL_freq': None,  # Non-reference warmup Richardson-Lucy update interval in solver iterations. None inherits warmup_RL_freq.

    # Output constraints, intensity baseline and input geometry
    'plot_every': 1000000000.0,  # Interval in solver iterations between progress plots.
    'average_img': 1,  # Number of lowest-error iterates averaged to return a stage field.
    'Fourier_last': True,  # Apply the Fourier constraint to the field returned by each retrieval stage.
    'final_fourier_constraint': not PARTIAL_COHERENCE,  # Enforce the Fourier constraint on the final joint reconstruction.
    'hologram_intensity_cutoff_vmin': -1,  # Intensity percentile used for baseline subtraction before taking amplitudes. Negative disables it.
    'binning': BINNING,  # Detector binning factor used to prepare the retrieval grid.
    'crop': CROP,  # Symmetric border crop in pixels. Zero disables cropping.
    'roi': None,  # Optional object-space region of interest supplied to the retrieval engine.

    # Modes and support initialization
    'Nmodes': len(MODES) if MODES is not None else 1,  # When modes is supplied, the engine uses len(modes).
    'modes': MODES,  # Support factors for individual modes. Factor 1 is the primary physical mode, and 0 is Gaussian.
    'gaussian_mode_sigma_x': 4.0,  # Gaussian residual-mode controls apply to a support-factor-zero mode.
    'gaussian_mode_sigma_y': 4.0,  # Gaussian residual mode width along y in retrieval-grid pixels.
    'gaussian_mode_angle': 0.0,  # Rotation of the Gaussian residual mode.
    'gaussian_mode_amplitude': 1.0,  # Amplitude of the Gaussian residual mode.
    'gaussian_mode_center': None,  # Optional center of the Gaussian residual mode. None uses the default center.
    'gaussian_mode_coherent': False,  # Include the Gaussian residual mode in the coherent sum.
    'mode_support_center': 'image',  # Center convention used to construct the scaled modal supports.
    'recenter_modal_supports': True,  # Shift scaled modal supports to the chosen support center.
    'mode_initialization': "support_fft",  # Method for seeding the modal Fourier fields, such as support_fft or random_phase.
    'mode_initialization_seed': 0,  # Random seed for modal field initialization.

    # Reference-gain fitting (single-energy only; disabled in this adapter)
    'fit_illumination_intensity': False,  # Cannot enable this single-energy gain fitter in the current multi-energy adapter.
    'align_global_phase': False,  # Same single-energy restriction; log-branch referencing below is a separate operation.
    'fit_aperture_illumination': False,  # Same single-energy restriction.
    'object_illumination_mode': 'pooled',  # Controls how illumination estimates are pooled in the reference-gain model.
    'aperture_factor_type': 'complex',  # Controls whether fitted aperture illumination factors are complex or amplitude-only.
    'illumination_intensity_reference_observation': 0,  # Zero-based observation index defining the incident intensity reference.

    # Physical mode groups and projection cadence
    'magnetization_mode_groups': None,  # Optional groups of support factors that share magnetic parameters.
    'illumination_mode_groups': None,  # Optional groups of support factors that share illumination parameters.
    'constrain_nonphysical_modes_common': True,  # Tie nonphysical modes to a common object for each energy and beam.
    'nonphysical_modes_common_relaxation': 1.0,  # Blend fraction for the common-object update of nonphysical modes.
    'projection_model': 'physical_factorized',  # Joint projection model. physical_factorized fits material, charge and magnetic terms.
    'projection_every': AUTO,  # AUTO = one complete observation sweep.
    'projection_start': None,  # None = first projection_every boundary.
    'projection_relaxation': 0.5,  # Blend fraction for physical projections during joint retrieval.
    'final_projection_relaxation': 0.5,  # Blend fraction for the final physical projection.
    'observation_weights': None,  # Optional per-observation fit weights. None gives equal weight.
    'rank_deficient': 'error',  # Policy for a rank-deficient projection fit. error raises an exception.
    'physical_iterations': 20,  # Number of alternating fit iterations per physical projection.

    # Magnetization, physical focus, material and thickness
    'clip_magnetization': False,  # Clamp fitted magnetization to the normalized interval from minus one to one.
    'magnetization_trend': 'none',  # Optional trend constraint on the magnetization across states. none disables it.
    'initial_binary_magnetization': False,  # Snap the initial magnetization estimate to the nearest allowed value.
    'binary_magnetization': False,  # Snap fitted magnetization maps to the nearest allowed value.
    'binary_magnetization_values': [-1.0, 0.0, 1.0],  # Allowed normalized magnetization values for snapping.
    'physical_projection_object_roi': True,  # Run the physical fit within the object region of interest.
    'physical_phase_reference': True,  # Use reference pixels to align phase branches before fitting log fields.
    'physical_spatial_weighting': None,  # Optional least-squares weighting from the mean reference exit-wave intensity.
    'projection_diagnostic_observation': None,  # Optional zero-based observation index for projection diagnostics.
    'projection_focus_prop_um': FOCUS["prop_dist"],  # Propagation distance in micrometres for focusing around the physical projection.
    'projection_focus_phase_rad': FOCUS["phase"],  # Global phase correction in radians for the physical projection focus transform.
    'projection_focus_setup': AUTO,  # AUTO binds raw CCD distance, pixel size * binning, reference energy and dx/dy / binning.
    'projection_focus_integer_wavelength': True,  # Round the focus propagation distance to an integer number of wavelengths.
    'material_mask': AUTO,  # AUTO = MATERIAL_REGION from this notebook; alternatively supply an explicit source-grid array.
    'material_thickness': INITIAL_MATERIAL_THICKNESS,  # None initializes positive thickness in the allowed region; explicit initial zeros exclude material.
    'fit_material_thickness': FIT_MATERIAL_THICKNESS,  # True lets thickness vary within the initially allowed positive domain.
    'saturated_states': None,  # Only supply states independently known to be saturated, e.g. {"sample": 1}.
    'zero_magnetization_outside_support': True,  # Force fitted magnetization to zero outside the real-space support.
    'zero_thickness_outside_support': True,  # Force fitted thickness to zero outside the real-space support.
    'projection_constraints_inside_support_only': True,  # Restrict joint field updates to the real-space support.
    'physical_constraints_inside_support_only': False,  # Legacy alias for restricting joint field updates to the support.
    'log_floor': 1e-12,  # Minimum field magnitude used when taking the complex logarithm.

    # Advanced chemical-species model controls
    'chemical_species': None,  # Advanced species models require species-aware result analysis; default notebook plots use one charge/magnetic response.
    'chemical_reference_mask': None,  # Optional mask identifying reference pixels for chemical-species fitting.
    'chemical_reference_alignment': True,  # Align chemical-model fields using the reference pixels before fitting.
    'chemical_initialization_seed': 0,  # Random seed for chemical-species model initialization.

    # Energy spectra, KK and known optical responses
    'charge_spectral_constraint': 'free',  # free, kk, known_beta, known_beta_kk (see engine spectral API).
    'magnetic_spectral_constraint': 'free',  # Same choices; do not impose KK without justified spectral coverage.
    'energy_values': AUTO,  # AUTO = unique energy labels in observation order; KK requires increasing energies.
    'known_charge_beta_spectrum': None,  # Optional tabulated charge absorptive index spectrum (beta).
    'known_charge_delta_spectrum': None,  # Optional tabulated charge dispersive index spectrum (delta).
    'known_magnetic_beta_spectrum': None,  # Optional tabulated magnetic absorptive index spectrum (beta).
    'known_magnetic_delta_spectrum': None,  # Optional tabulated magnetic dispersive index spectrum (delta).
    'charge_absorption_part': 'real',  # Complex component treated as absorption in the charge response.
    'magnetic_absorption_part': 'real',  # Complex component treated as absorption in the magnetic response.
    'charge_response_real_range': None,  # Optional bounds on the real part of the fitted charge response.
    'charge_response_imag_range': None,  # Optional bounds on the imaginary part of the fitted charge response.
    'magnetic_response_real_range': None,  # Optional bounds on the real part of the fitted magnetic response.
    'magnetic_response_imag_range': None,  # Optional bounds on the imaginary part of the fitted magnetic response.
    'kk_sign': 1.0,  # Sign convention applied to the Kramers-Kronig transform.
    'kk_subtract_baseline': True,  # Remove the baseline from the input to the Kramers-Kronig transform.
    'kk_normalize_input': False,  # Normalize the input to the Kramers-Kronig transform.
    'known_spectrum_normalization': 'none',  # Normalization applied to supplied charge and magnetic reference spectra.
    'fit_known_spectrum_scale': True,  # Fit a multiplicative scale for the supplied physical reference spectra.
    'fit_known_spectrum_offset': True,  # Fit an additive offset for the supplied physical reference spectra.

    # Alternative engine projection families (inactive with physical_factorized)
    'rank': 1,  # Number of components retained by the alternative low-rank projection.
    'projection_static_mode': 'mean',  # Method for estimating the common static component in alternative projections.
    'spectral_constraint': 'free',  # Spectral model selection for the alternative projection families.
    'known_beta_spectrum': None,  # Optional tabulated absorptive spectrum for an alternative spectral projection.
    'known_delta_spectrum': None,  # Optional tabulated dispersive spectrum for an alternative spectral projection.
    'absorption_part': 'real',  # Complex response component treated as absorption in alternative spectral projections.
    'known_beta_normalization': 'none',  # Normalization applied to the alternative projection's supplied beta spectrum.
    'fit_known_beta_scale': True,  # Fit a multiplicative scale for the alternative projection's supplied beta spectrum.
    'fit_known_beta_offset': True,  # Fit an additive offset for the alternative projection's supplied beta spectrum.

    # Physical optical-index products and independent thickness anchors
    'charge_kt_delta_range': None,  # Optional bounds on the fitted charge k × thickness × delta product.
    'charge_kt_beta_range': None,  # Optional bounds on the fitted charge k × thickness × beta product.
    'magnetic_kt_delta_range': None,  # Optional bounds on the fitted magnetic k × thickness × delta product.
    'magnetic_kt_beta_range': None,  # Optional bounds on the fitted magnetic k × thickness × beta product.
    'wave_numbers': None,  # Optional inverse-metre wave numbers for externally calibrated optical indices.
    'thickness': None,  # Optional independently known physical thickness in metres; distinct from the fitted relative map.
    'known_charge_kt_beta_spectrum': None,  # Optional tabulated charge k × thickness × beta product.
    'known_charge_kt_delta_spectrum': None,  # Optional tabulated charge k × thickness × delta product.
    'known_magnetic_kt_beta_spectrum': None,  # Optional tabulated magnetic k × thickness × beta product.
    'known_magnetic_kt_delta_spectrum': None,  # Optional tabulated magnetic k × thickness × delta product.
}
# Coverage audit only: this does not insert values from defaults.
assert set(RECIPE_PARAMETERS) == set(hs.engine.default_universal_phase_retrieval_recipe())
print("Explicit recipe controls:", len(RECIPE_PARAMETERS))


## Inspect the raw detector and prepare the dataset

The raw preview shows the independently specified detector center. Energy and detector distance
are read directly from NeXus unless explicitly overridden above; pixel size is specified here.
Dark fitting uses `DARK_FIT_RECIPE`. The fixed detector PNG is disabled. Beamstop masks remain
exclusions, with no invented intensity underneath them. The loader centers raw intensities and
masks together and maps energies onto a common reciprocal-space grid with flux conservation.

This uses small-angle planar-detector geometry. Set per-pair centers for lateral drift or camera
changes. The support coordinates belong to the centered object grid at the reference energy.


In [ ]:
raw_loader = hs.StreamingSextantsLoader(RAW_FOLDER)
preview_frame = raw_loader.load(hs.first_id(PAIRS[0]["plus"]))
print("First raw geometry:", {key: preview_frame.metadata.get(key) for key in ("energy_eV", "ccd_dist_m", "polarization_code", "frame_count")})
preview_mask_file = MASK_FOLDER / f"mask_beamstop_{BEAMSTOP_MASK_IDS['+']}.png"
preview_mask = hs.load_red_mask_png(preview_mask_file, preview_frame.image.shape)
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(np.log1p(np.maximum(preview_frame.image, 0)), cmap="magma")
ax.contour(preview_mask, levels=[0.5], colors="cyan", linewidths=0.5)
ax.plot(DETECTOR_CENTER[1], DETECTOR_CENTER[0], "+", color="lime", markersize=14)
ax.set_title("Raw detector: selected center (green) and beamstop mask (cyan)")
plt.show()

support = None if SUPPORT_PNG is None else hs.load_red_mask_png(SUPPORT_PNG, preview_frame.image.shape)
del preview_frame

dataset = hs.load_energy_scan(
    PAIRS, RAW_FOLDER, MASK_FOLDER,
    detector_center=DETECTOR_CENTER, pixel_size_m=PIXEL_SIZE_M,
    reference_energy_eV=REFERENCE_ENERGY_EV, reference_distance_m=REFERENCE_DISTANCE_M,
    focus=FOCUS, dark_fit_recipe=DARK_FIT_RECIPE,
    beamstop_mask_ids=BEAMSTOP_MASK_IDS, dark_ids=DARK_IDS,
    supportmask=support, support_coordinates=SUPPORT_COORDINATES if support is None else None,
    energy_tolerance_eV=ENERGY_TOLERANCE_EV,
    exposure_normalize=EXPOSURE_NORMALIZE, intensity_offset=INTENSITY_OFFSET,
    common_q=COMMON_Q_GRID, dtype=RAW_DTYPE,
)
dataset["preprocessing"].update(
    configuration_source="standalone notebook + raw NeXus metadata",
    support_file=str(SUPPORT_PNG) if SUPPORT_PNG is not None else None,
    support_coordinates=SUPPORT_COORDINATES if SUPPORT_PNG is None else None,
)
recipe = hs.resolve_hyperspectral_recipe(dataset, RECIPE_PARAMETERS, material_region=MATERIAL_REGION)
print("Intensity stack:", dataset["holograms"].shape)
print("Geometry from raw files/settings:", dataset["experimental_setup"])
print("Input arrays (GiB):", (dataset["holograms"].nbytes + dataset["mask_pixel"].nbytes) / 2**30)
print("Retrieval also needs complex field stacks for every mode and observation.")
if SAVE_PREPARED:
    hs.save_result(dataset, PREPARED_H5, overwrite=OVERWRITE)
    print("Prepared cache:", PREPARED_H5)
# Optional reuse of this notebook's own cache:
# dataset = wf.load_data_dict(PREPARED_H5)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for index in (0, 1):
    row = dataset["records"][index]
    axes[index, 0].imshow(np.log1p(np.maximum(dataset["holograms"][index], 0)), cmap="magma")
    axes[index, 0].set_title(f"{row['label']} {row['energy_eV']} eV: log(1 + intensity)")
    axes[index, 1].imshow(dataset["mask_pixel"][index], cmap="gray", vmin=0, vmax=1)
    axes[index, 1].set_title("Excluded detector pixels")
axes[0, 2].imshow(dataset["supportmask"], cmap="gray")
axes[0, 2].set_title("Allowed object support")
material = recipe["material_mask"]
if material is None:
    material = (dataset["supportmask"] if recipe["zero_thickness_outside_support"] else np.ones_like(dataset["supportmask"]))
axes[1, 2].imshow(material, cmap="gray")
axes[1, 2].set_title("Allowed thickness-fit region")
fig.tight_layout()
plt.show()

# Standalone FTH-style preview to inspect center/focus against the support before retrieval.
valid_pair = (dataset["mask_pixel"][0] == 0) & (dataset["mask_pixel"][1] == 0)
preview_hologram = (dataset["holograms"][0] - dataset["holograms"][1]) * valid_pair
preview_setup = recipe["projection_focus_setup"]
if preview_setup is not None:
    # FTH preview uses the source grid; recipe focus geometry is on the binned retrieval grid.
    preview_setup = dict(preview_setup,
        px_size=preview_setup["px_size"]/recipe["binning"],
        dx=preview_setup.get("dx",0)*recipe["binning"],
        dy=preview_setup.get("dy",0)*recipe["binning"])
preview_field = hs.engine._projection_focus_transform(
    preview_hologram[None], [dataset["records"][0]["energy_eV"]],
    recipe["projection_focus_prop_um"], recipe["projection_focus_phase_rad"], preview_setup,
    integer_wavelength=recipe["projection_focus_integer_wavelength"])
preview_object = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(preview_field[0])))
fig, ax = plt.subplots(figsize=(7, 6))
vmin, vmax = wf.finite_percentile_limits(preview_object.real, (1,99))
ax.imshow(preview_object.real, cmap="gray", vmin=vmin, vmax=vmax)
ax.contour(dataset["supportmask"], levels=[0.5], colors="red", linewidths=.6)
ax.set_title("First-pair FTH preview with this notebook's focus and support")
plt.show()


## Joint physical reconstruction

Each joint sweep updates all energies and both helicities before applying the shared physical
model. Geometry, focus and preprocessing come only from the settings above and raw metadata.
Incoming-flux monitor values can be supplied per pair; exposure normalization alone is not a
measurement of flux. The engine's single-energy reference-gain correction stays disabled.

With `FIT_MATERIAL_THICKNESS=True` and `MATERIAL_REGION="support"`, positive thickness is
initialized throughout all support apertures. The fit can reduce thickness to zero rather
than having the sample/reference distinction imposed in advance. Outside the support it stays
zero. Initial zero-thickness pixels are excluded from the allowed material domain, so use a
positive initial map throughout any region whose material occupancy should be fitted.

Thickness, spectral responses and illumination have scale/gauge ambiguities; this is a relative
fit, not guaranteed unique material segmentation. Optional KK constraints need justified spectral
coverage. The phase-reference option selects complex-log branches, not absolute phase.


In [ ]:
# The complete recipe was defined at the beginning and resolved after raw loading.
# No parameter values or schedules are changed here.
hs.engine.print_universal_workflow(recipe, state_labels=[r["state"] for r in dataset["records"]])


In [ ]:
# This notebook's propagation, phase and dx/dy act reversibly around the physical fit.
# The local engine accounts for the common-q grid's energy-dependent pixel angle.
result = hs.reconstruct(dataset, recipe)
components = result["components"]
result["detector_relative_error"] = hs.detector_relative_errors(dataset, result)
print("Projection model:", components["projection_model"])
print("Physical fit residual:", components.get("fit_residual_rms"))
print("Identifiability/gauge diagnostics:", components.get("identifiable"))
print("Field stack:", result["fields"].shape)


## Inspect the hyperspectral result and save

Spatial physical components use the engine's unshifted object convention. The plots
below shift them to the centered display frame used for support drawing. Detector
fields remain in centered detector coordinates. Inspect the other modes and detector
errors before assigning physical meaning to a feature in mode 0 alone.


In [ ]:
energy = np.asarray(components["energy_names"], dtype=float)
order = np.argsort(energy)
magnetization = np.fft.fftshift(np.asarray(components["magnetization"]), axes=(-2, -1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
image = axes[0].imshow(magnetization[0], cmap="RdBu_r")
axes[0].set_title("Shared relative magnetization")
fig.colorbar(image, ax=axes[0])
for axis, key in zip(axes[1:], ("charge_response", "magnetic_response")):
    response = np.asarray(components[key])
    axis.plot(energy[order], response.real[order], "o-", label="real")
    axis.plot(energy[order], response.imag[order], "o-", label="imaginary")
    axis.set(xlabel="Energy (eV)", ylabel="Relative log response", title=key)
    axis.legend(); axis.grid(alpha=.3)
fig.tight_layout()
FIGURE_PNG.parent.mkdir(parents=True, exist_ok=True)
if FIGURE_PNG.exists() and not OVERWRITE:
    raise FileExistsError(f"Choose a new FIGURE_PNG or set OVERWRITE=True: {FIGURE_PNG}")
fig.savefig(FIGURE_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for label in ("+", "-"):
    indices = [i for i, r in enumerate(result["records"]) if r["label"] == label]
    ax.plot([result["records"][i]["energy_eV"] for i in indices],
            result["detector_relative_error"][indices], "o-", label=label)
ax.set(xlabel="Energy (eV)", ylabel="Relative measured-intensity residual",
       title="Forward-model agreement, including blur when enabled")
ax.legend(); ax.grid(alpha=.3); fig.tight_layout(); plt.show()

# Same physical-projection plane as the fitted components, mode 0 only for inspection.
fields = result["fields"]
primary = fields[:, 0] if fields.ndim == 4 else fields
focused = hs.engine._projection_focus_transform(
    primary, [r["energy_eV"] for r in result["records"]],
    recipe["projection_focus_prop_um"], recipe["projection_focus_phase_rad"],
    recipe["projection_focus_setup"], integer_wavelength=recipe["projection_focus_integer_wavelength"])
objects = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(focused, axes=(-2, -1))), axes=(-2, -1))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for index, axis in enumerate(axes):
    axis.imshow(np.abs(objects[index]), cmap="gray")
    axis.set_title(f"First energy, {result['records'][index]['label']}: primary amplitude")
fig.tight_layout(); plt.show()

result["figure_png"] = str(FIGURE_PNG)
hs.save_result(result, RESULT_H5, overwrite=OVERWRITE)
print("Saved:", RESULT_H5)
print("Figure:", FIGURE_PNG)
